## 🎯 Learning Objectives
* Understand the concept and importance of dynamic graph construction in LangGraph for building adaptive AI agents.
* Learn how to programmatically define and modify LangGraph structures based on runtime conditions or inputs.
* Implement a practical example demonstrating conditional node and edge addition to a LangGraph.
* Analyze the performance implications and identify suitable use cases for dynamic graph construction.


## Dynamic Graph Construction at Runtime with LangGraph

In the realm of advanced AI agents, adaptability is paramount. Static, pre-defined workflows, while robust for predictable tasks, often fall short when agents need to respond to highly variable environments, user inputs, or internal states. This is where **dynamic graph construction at runtime** becomes a game-changer.

### What is Dynamic Graph Construction?

Imagine you're building a highly modular robot. Instead of having a fixed set of tools and a rigid sequence of operations, this robot can analyze its current task and environment, then *assemble* the necessary tools and *design* a custom workflow on the fly. If it needs to pick up a delicate object, it might attach a soft gripper and plan a gentle movement sequence. If it needs to drill, it attaches a drill and plans a different, more forceful sequence. The robot's internal 'workflow graph' isn't fixed; it's built dynamically based on the situation.

In LangGraph, dynamic graph construction refers to the ability to programmatically define the structure of your agent's workflow – its nodes and edges – based on conditions that are only known at the time the agent is invoked or even during its execution. Instead of hardcoding every possible path, you write logic that *builds* the graph's topology based on initial inputs, environmental observations, or internal decision-making.

### Why is it Crucial for Advanced AI Agents?

1.  **Adaptability**: Agents can tailor their behavior to specific tasks, user queries, or environmental changes without needing a massive, complex, and often inefficient monolithic graph that tries to cover all possibilities.
2.  **Resource Optimization**: Only the necessary components (nodes) are included in the workflow, preventing the execution of irrelevant or costly operations.
3.  **Enhanced Decision-Making**: The agent's initial analysis can directly influence its subsequent architecture, allowing for more sophisticated and context-aware planning.
4.  **Modularity and Reusability**: You can define reusable 'sub-graphs' or 'modules' that are conditionally integrated into a larger workflow, promoting cleaner code and easier maintenance.
5.  **Self-Modifying Agents**: In extreme cases, agents could even learn to modify their own internal structure over time, evolving their capabilities based on experience (though this is a more advanced concept building on dynamic construction).

### How LangGraph Facilitates This

While LangGraph's `Graph().compile()` method creates an immutable execution graph, the *definition* of that `Graph` object can be entirely programmatic. This means you can write a function that takes initial parameters (e.g., a user query, a configuration object) and *returns* a fully constructed `Graph` instance with nodes and edges added conditionally. This graph is then compiled and executed. This allows for dynamic behavior *before* the graph is compiled, effectively creating a custom workflow for each specific invocation.

Let's explore this with an example where an agent decides whether to include a 'code generation and execution' module based on the complexity of the user's query.


In [ ]:
import operator
from typing import Annotated, List, Tuple, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, StateGraph

# --- 1. Define the Agent State ---
# This state will be passed between nodes and updated.
class AgentState(TypedDict):
    query: str
    analysis_result: str
    generated_code: str
    execution_output: str
    final_response: str
    # The 'messages' key is often useful for conversational agents
    messages: Annotated[List[BaseMessage], operator.add]

# --- 2. Define Node Functions ---
# These are the building blocks of our graph.

def initial_analysis_node(state: AgentState) -> AgentState:
    """Analyzes the query to determine if code generation/execution is needed."""
    query = state["query"]
    print(f"[Initial Analysis] Analyzing query: '{query}'")
    
    # Simple heuristic: if 'code', 'execute', 'script', 'program' are in query,
    # assume code generation is needed.
    if any(keyword in query.lower() for keyword in ["code", "execute", "script", "program"]):
        analysis_result = "CODE_NEEDED"
        print("[Initial Analysis] Code generation identified as necessary.")
    else:
        analysis_result = "INFO_RETRIEVAL_NEEDED"
        print("[Initial Analysis] Information retrieval identified as sufficient.")
        
    return {"analysis_result": analysis_result, "messages": [HumanMessage(content=f"Analysis: {analysis_result}")]}

def code_generator_node(state: AgentState) -> AgentState:
    """Generates placeholder code based on the query."""
    query = state["query"]
    print(f"[Code Generator] Generating code for query: '{query}'")
    # In a real scenario, this would use an LLM to generate actual code.
    generated_code = f"print('Executing code for: {query}')\nresult = 'Dynamic code output for {query}'"
    return {"generated_code": generated_code, "messages": [HumanMessage(content=f"Generated code: {generated_code}")]}

def code_executor_node(state: AgentState) -> AgentState:
    """Executes the generated code."""
    generated_code = state["generated_code"]
    print(f"[Code Executor] Executing code:\n{generated_code}")
    execution_output = ""
    try:
        # For demonstration, we'll just 'simulate' execution.
        # In a real scenario, you'd use a tool executor or sandbox.
        exec(generated_code, globals(), locals())
        execution_output = locals().get('result', 'Code executed successfully.')
    except Exception as e:
        execution_output = f"Error during code execution: {e}"
    return {"execution_output": execution_output, "messages": [HumanMessage(content=f"Execution output: {execution_output}")]}

def info_retriever_node(state: AgentState) -> AgentState:
    """Retrieves information based on the query."""
    query = state["query"]
    print(f"[Info Retriever] Retrieving information for query: '{query}'")
    # In a real scenario, this would involve RAG, database lookup, etc.
    retrieved_info = f"Information about '{query}': This is a dynamically retrieved piece of data."
    return {"final_response": retrieved_info, "messages": [HumanMessage(content=f"Retrieved info: {retrieved_info}")]}

def final_response_node(state: AgentState) -> AgentState:
    """Formulates the final response based on all gathered information."""
    query = state["query"]
    analysis_result = state.get("analysis_result", "N/A")
    generated_code = state.get("generated_code", "N/A")
    execution_output = state.get("execution_output", "N/A")
    retrieved_info = state.get("final_response", "N/A") # Reusing final_response key for info

    print(f"[Final Response] Compiling response for query: '{query}'")
    
    response_parts = [
        f"Query: {query}",
        f"Analysis: {analysis_result}"
    ]
    
    if analysis_result == "CODE_NEEDED":
        response_parts.append(f"Generated Code: {generated_code}")
        response_parts.append(f"Execution Output: {execution_output}")
    else:
        response_parts.append(f"Retrieved Information: {retrieved_info}")
        
    final_response_text = "\n".join(response_parts)
    print(f"[Final Response] Final response generated.")
    return {"final_response": final_response_text, "messages": [HumanMessage(content=f"Final Response: {final_response_text}")]}

# --- 3. Dynamic Graph Construction Function ---
def build_dynamic_agent_graph(initial_query: str):
    """Constructs and compiles a LangGraph based on the initial query."""
    workflow = StateGraph(AgentState)

    # Always add the initial analysis and final response nodes
    workflow.add_node("initial_analysis", initial_analysis_node)
    workflow.add_node("final_response", final_response_node)

    # Determine if code generation/execution path is needed based on initial query
    # This is the 'dynamic construction' part.
    code_path_needed = any(keyword in initial_query.lower() for keyword in ["code", "execute", "script", "program"])

    if code_path_needed:
        print(f"\n--- Building graph with CODE GENERATION path for query: '{initial_query}' ---")
        workflow.add_node("code_generator", code_generator_node)
        workflow.add_node("code_executor", code_executor_node)
        
        # Define edges for the code path
        workflow.add_edge("initial_analysis", "code_generator")
        workflow.add_edge("code_generator", "code_executor")
        workflow.add_edge("code_executor", "final_response")
        
        # Set entry point
        workflow.set_entry_point("initial_analysis")
        # Set exit point
        workflow.set_finish_point("final_response")

    else:
        print(f"\n--- Building graph with INFO RETRIEVAL path for query: '{initial_query}' ---")
        workflow.add_node("info_retriever", info_retriever_node)
        
        # Define edges for the info retrieval path
        workflow.add_edge("initial_analysis", "info_retriever")
        workflow.add_edge("info_retriever", "final_response")
        
        # Set entry point
        workflow.set_entry_point("initial_analysis")
        # Set exit point
        workflow.set_finish_point("final_response")

    # Compile the graph
    app = workflow.compile()
    print("Graph compiled successfully.")
    return app

# --- 4. Demonstrate Dynamic Graph Execution ---

# Scenario 1: Query requiring code execution
print("\n=== Running Agent for Code Execution Query ===")
query_code = "Please write and execute a Python script to greet the user."
agent_code = build_dynamic_agent_graph(query_code)

initial_state_code = {"query": query_code, "messages": [HumanMessage(content=query_code)]}
result_code = agent_code.invoke(initial_state_code)
print("\n--- Final Result (Code Path) ---")
print(result_code["final_response"])

# Scenario 2: Query requiring simple information retrieval
print("\n=== Running Agent for Information Retrieval Query ===")
query_info = "What is the capital of France?"
agent_info = build_dynamic_agent_graph(query_info)

initial_state_info = {"query": query_info, "messages": [HumanMessage(content=query_info)]}
result_info = agent_info.invoke(initial_state_info)
print("\n--- Final Result (Info Path) ---")
print(result_info["final_response"])

# Scenario 3: Another code execution query
print("\n=== Running Agent for Another Code Execution Query ===")
query_script = "Can you execute a program to calculate 2+2?"
agent_script = build_dynamic_agent_graph(query_script)

initial_state_script = {"query": query_script, "messages": [HumanMessage(content=query_script)]}
result_script = agent_script.invoke(initial_state_script)
print("\n--- Final Result (Script Path) ---")
print(result_script["final_response"])


### Interpreting the Code Output and Performance Trade-offs

The code demonstrates how a single function, `build_dynamic_agent_graph`, can construct entirely different LangGraph workflows based on an initial input (`initial_query`).

**Interpretation of Output:**

*   **Scenario 1 (Code Execution Query)**: When the query contains keywords like "code" or "execute", the `build_dynamic_agent_graph` function detects this. It then proceeds to add `code_generator` and `code_executor` nodes to the workflow, along with the necessary edges to connect them in sequence: `initial_analysis` -> `code_generator` -> `code_executor` -> `final_response`. The output clearly shows the execution of these specific nodes, including the simulated code generation and execution steps.
*   **Scenario 2 (Information Retrieval Query)**: For queries without the specified keywords, the function constructs a simpler graph. It adds only the `info_retriever` node, connecting the workflow as: `initial_analysis` -> `info_retriever` -> `final_response`. The output reflects this streamlined path, skipping the code-related nodes entirely.
*   **Scenario 3 (Another Code Execution Query)**: This further reinforces the dynamic nature, showing that any query matching the criteria will trigger the code-execution path.

This approach allows the agent to be highly responsive and efficient, only including the necessary computational steps for a given task.

**Performance Trade-offs:**

**Advantages:**

1.  **Efficiency**: By only including relevant nodes and edges, you avoid unnecessary computations and resource allocation. For example, if a query doesn't require complex data analysis, those expensive LLM calls or tool invocations are simply not added to the graph.
2.  **Scalability**: As your agent's capabilities grow, you can add new modules (nodes) and conditionally integrate them without making the core graph definition unwieldy or forcing all invocations through a bloated workflow.
3.  **Flexibility**: Agents can adapt to a wider range of tasks and contexts, leading to more robust and intelligent behavior.
4.  **Maintainability**: Breaking down complex logic into smaller, conditionally assembled graphs can make individual components easier to understand, test, and debug.

**Disadvantages:**

1.  **Increased Complexity in Graph Definition**: The logic for *building* the graph itself becomes more complex. You need robust conditional statements and careful management of node and edge additions.
2.  **Debugging Challenges**: While individual sub-graphs might be simpler, debugging the graph *construction logic* can be harder, as the graph's structure changes with inputs. Visualizing the graph for every possible dynamic configuration can be cumbersome.
3.  **Compilation Overhead**: Each time a new graph structure is generated (even if it's just a slight variation), LangGraph needs to compile it. For extremely high-throughput systems where graph structures change very frequently, this compilation overhead *could* become a factor, though for most agentic applications, it's negligible compared to LLM inference times.
4.  **State Management**: Ensuring that the `AgentState` correctly flows through dynamically constructed paths requires careful design.

**Typical Use Cases:**

*   **Adaptive Planning Agents**: An agent that plans its next steps based on the current environment state, dynamically adding or removing planning sub-graphs.
*   **Multi-tool Agents**: Agents that select and integrate specific tools (represented as nodes) into their workflow based on the user's request or internal reasoning.
*   **Self-Correction/Refinement Loops**: An agent that, upon detecting an error or low confidence, dynamically inserts a 'refinement' or 'fact-checking' sub-graph into its ongoing process.
*   **Personalized Workflows**: Agents that tailor their process based on user preferences, historical data, or user roles.
*   **Complex Orchestration**: A supervisor agent that dynamically composes sub-agents (each represented by a sub-graph) into a larger workflow based on the overall task requirements.

By mastering dynamic graph construction, you unlock the full potential of LangGraph to build truly intelligent, adaptable, and efficient AI systems.


### Resources

*   **LangGraph Documentation**: The official documentation is always the best place to start for in-depth understanding of `StateGraph`, nodes, and edges.
    *   [LangGraph Introduction](https://langchain-ai.github.io/langgraph/)
    *   [LangGraph StateGraph](https://langchain-ai.github.io/langgraph/reference/graphs/state_graph/)
    *   [LangGraph Tutorials](https://langchain-ai.github.io/langgraph/tutorials/)
*   **LangChain Expression Language (LCEL)**: Understanding LCEL is crucial as LangGraph builds upon it for defining runnable components.
    *   [LCEL Documentation](https://python.langchain.com/docs/expression_language/)
*   **Advanced Agent Architectures**: Explore articles and research papers on adaptive and self-modifying AI systems for broader context.
    *   Look for recent publications on 'adaptive agents', 'meta-learning for agents', or 'dynamic workflow orchestration in AI'.
